In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import yfinance as yf

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

In [2]:
# Example risk-factor price history
risk_factor_prices = pd.DataFrame({
    "EURUSD": [1.10, 1.105, 1.102, 1.108, 1.104],
    "GBPUSD": [1.25, 1.255, 1.252, 1.260, 1.257]
})

# Portfolio sensitivities to each risk factor
sensitivities = pd.Series({
    "EURUSD": 1_000_000,
    "GBPUSD": 500_000
})

risk_factor_prices

,EURUSD,GBPUSD
0,1.100,1.250
1,1.105,1.255
2,1.102,1.252
3,1.108,1.260
4,1.104,1.257


In [3]:
sensitivities

,0
EURUSD,1000000
GBPUSD,500000


In [4]:
returns = risk_factor_prices.pct_change().dropna()

returns

,EURUSD,GBPUSD
1,0.004545,0.004000
2,-0.002715,-0.002390
3,0.005445,0.006390
4,-0.003610,-0.002381


In [5]:
portfolio_pnl = returns @ sensitivities

portfolio_pnl

,0
1,6545.454545
2,-3910.151250
3,8639.534277
4,-4800.584494


In [6]:
confidence = 0.95

pnl_quantile = portfolio_pnl.quantile(1 - confidence)

pnl_quantile

np.float64(-4667.019507196982)

In [7]:
historical_var = abs(pnl_quantile)

historical_var

np.float64(4667.019507196982)

In [8]:
tail_losses = portfolio_pnl[portfolio_pnl <= pnl_quantile]

tail_losses

,0
4,-4800.584494


In [9]:
historical_es = abs(tail_losses.mean())

historical_es

np.float64(4800.5844937253905)

In [10]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import yfinance as yf

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter


class RiskEngine:
    """
    Reusable market risk engine for sensitivity-based portfolios.

    The engine calculates Value at Risk (VaR), Expected Shortfall (ES),
    and simple stress losses using Historical, Parametric, and
    Monte Carlo methodologies.

    The portfolio is represented using linear sensitivities to risk
    factors. Portfolio P&L is therefore approximated as:

        P&L = risk-factor returns @ sensitivities

    This is a linear sensitivity-based risk engine and does not perform
    full revaluation of nonlinear instruments.

    Parameters
    ----------
    sensitivities : pandas.Series
        Portfolio sensitivity for each risk factor.

    risk_factor_prices : pandas.DataFrame
        Historical prices or levels for the corresponding risk factors.

    Notes
    -----
    - Risk-factor names in sensitivities and risk_factor_prices must match.
    - Market data must be numeric and finite.
    - Historical VaR/ES use observed historical returns.
    - Parametric VaR/ES assume normally distributed returns.
    - Monte Carlo uses a multivariate normal return model.
    - Monte Carlo uses a random seed for reproducibility.
    - Stress loss currently applies a common return shock to all factors.
    """

    def __init__(self, sensitivities, risk_factor_prices):
        self.sensitivities = sensitivities
        self.risk_factor_prices = risk_factor_prices

        self._validate_inputs()

    # ---------------------------------------------------------
    # Input Validation
    # ---------------------------------------------------------

    def _validate_inputs(self):
        """
        Validate portfolio sensitivities and risk-factor market data.

        Checks input types, numeric values, finite values, matching
        risk-factor names, and duplicate risk-factor labels.
        """

        if not isinstance(self.sensitivities, pd.Series):
            raise TypeError(
                "sensitivities must be a pandas Series."
            )

        if not isinstance(self.risk_factor_prices, pd.DataFrame):
            raise TypeError(
                "risk_factor_prices must be a pandas DataFrame."
            )

        if not pd.api.types.is_numeric_dtype(self.sensitivities):
            raise TypeError(
                "Sensitivities must contain numeric values."
            )

        if not all(
            pd.api.types.is_numeric_dtype(dtype)
            for dtype in self.risk_factor_prices.dtypes
        ):
            raise TypeError(
                "Risk factor prices must contain numeric values."
            )

        if not np.isfinite(self.sensitivities.to_numpy()).all():
            raise ValueError(
                "Sensitivities must contain only finite values."
            )

        if not np.isfinite(
            self.risk_factor_prices.to_numpy()
        ).all():
            raise ValueError(
                "Risk factor prices must contain only finite values."
            )

        if set(self.sensitivities.index) != set(
            self.risk_factor_prices.columns
        ):
            raise ValueError(
                "Risk factors in sensitivities and "
                "risk_factor_prices must match."
            )

        if self.sensitivities.index.duplicated().any():
            raise ValueError(
                "Sensitivities must not contain duplicate "
                "risk-factor names."
            )

        if self.risk_factor_prices.columns.duplicated().any():
            raise ValueError(
                "Risk factor prices must not contain duplicate "
                "risk-factor names."
            )

    # ---------------------------------------------------------
    # Parameter Validation
    # ---------------------------------------------------------

    def _validate_confidence(self, confidence):
        """Validate that confidence is strictly between 0 and 1."""

        if not isinstance(confidence, (int, float)):
            raise ValueError(
                "Confidence must be a numeric value between 0 and 1."
            )

        if not 0 < confidence < 1:
            raise ValueError(
                "Confidence must be between 0 and 1."
            )

    def _validate_horizon(self, horizon):
        """Validate that the risk horizon is a positive integer."""

        if (
            isinstance(horizon, bool)
            or not isinstance(horizon, int)
            or horizon <= 0
        ):
            raise ValueError(
                "Horizon must be a positive integer."
            )

    def _validate_simulations(self, simulations):
        """Validate that the Monte Carlo simulation count is positive."""

        if (
            isinstance(simulations, bool)
            or not isinstance(simulations, int)
            or simulations <= 0
        ):
            raise ValueError(
                "Simulations must be a positive integer."
            )

    # ---------------------------------------------------------
    # Return Calculation
    # ---------------------------------------------------------

    def _calculate_returns(self, horizon):
        """
        Calculate percentage returns over the specified horizon.

        Parameters
        ----------
        horizon : int
            Number of periods used to calculate returns.

        Returns
        -------
        pandas.DataFrame
            Historical percentage returns for each risk factor.
        """

        self._validate_horizon(horizon)

        if len(self.risk_factor_prices) <= horizon:
            raise ValueError(
                f"Insufficient historical observations for "
                f"a {horizon}-period horizon."
            )

        returns = self.risk_factor_prices.pct_change(
            periods=horizon,
            fill_method=None
        ).dropna()

        if returns.empty:
            raise ValueError(
                f"Insufficient historical observations for "
                f"a {horizon}-period horizon."
            )

        return returns

    # ---------------------------------------------------------
    # Portfolio P&L
    # ---------------------------------------------------------

    def _calculate_portfolio_pnl(self, returns):
        """
        Calculate portfolio P&L from risk-factor returns.

        The calculation uses a linear sensitivity approximation:

            P&L = returns @ sensitivities
        """

        aligned_sensitivities = self.sensitivities[
            returns.columns
        ]

        pnl = returns @ aligned_sensitivities

        return pnl

    # ---------------------------------------------------------
    # Historical VaR
    # ---------------------------------------------------------

    def historical_var(self, confidence=0.95, horizon=1):
        """
        Calculate Historical Value at Risk.

        Historical VaR is calculated as the absolute value of the
        lower-tail percentile of the historical portfolio P&L
        distribution.

        Parameters
        ----------
        confidence : float, default=0.95
            VaR confidence level.

        horizon : int, default=1
            Return horizon in periods.

        Returns
        -------
        float
            Historical VaR.
        """

        self._validate_confidence(confidence)
        self._validate_horizon(horizon)

        returns = self._calculate_returns(horizon)
        pnl = self._calculate_portfolio_pnl(returns)

        var_threshold = pnl.quantile(1 - confidence)

        var = abs(var_threshold)

        return var

    # ---------------------------------------------------------
    # Historical Expected Shortfall
    # ---------------------------------------------------------

    def historical_es(self, confidence=0.95, horizon=1):
        """
        Calculate Historical Expected Shortfall.

        Expected Shortfall is the average portfolio loss in the
        observations falling beyond the Historical VaR threshold.

        Parameters
        ----------
        confidence : float, default=0.95
            ES confidence level.

        horizon : int, default=1
            Return horizon in periods.

        Returns
        -------
        float
            Historical Expected Shortfall.
        """

        self._validate_confidence(confidence)
        self._validate_horizon(horizon)

        returns = self._calculate_returns(horizon)
        pnl = self._calculate_portfolio_pnl(returns)

        var_threshold = pnl.quantile(1 - confidence)

        tail_losses = pnl[pnl <= var_threshold]

        if tail_losses.empty:
            raise ValueError(
                "No observations available in the Expected "
                "Shortfall tail."
            )

        es = abs(tail_losses.mean())

        return es

    # ---------------------------------------------------------
    # Parametric VaR
    # ---------------------------------------------------------

    def parametric_var(self, confidence=0.95, horizon=1):
        """
        Calculate Parametric VaR under a normal-return assumption.

        Portfolio volatility is estimated from the historical
        covariance matrix of risk-factor returns and scaled by
        the square root of the selected horizon.

        Parameters
        ----------
        confidence : float, default=0.95
            VaR confidence level.

        horizon : int, default=1
            Return horizon in periods.

        Returns
        -------
        float
            Parametric VaR.
        """

        self._validate_confidence(confidence)
        self._validate_horizon(horizon)

        returns = self._calculate_returns(1)

        covariance_matrix = returns.cov()

        aligned_sensitivities = self.sensitivities[
            covariance_matrix.columns
        ]

        portfolio_variance = (
            aligned_sensitivities.T
            @ covariance_matrix
            @ aligned_sensitivities
        )

        portfolio_volatility = np.sqrt(
            portfolio_variance
        )

        z_score = norm.ppf(1 - confidence)

        horizon_volatility = (
            portfolio_volatility * np.sqrt(horizon)
        )

        var = abs(z_score * horizon_volatility)

        return var

    # ---------------------------------------------------------
    # Parametric Expected Shortfall
    # ---------------------------------------------------------

    def parametric_es(self, confidence=0.95, horizon=1):
        """
        Calculate Parametric Expected Shortfall.

        Expected Shortfall is derived from the normal distribution
        using the portfolio volatility and the selected confidence
        level.

        Parameters
        ----------
        confidence : float, default=0.95
            ES confidence level.

        horizon : int, default=1
            Return horizon in periods.

        Returns
        -------
        float
            Parametric Expected Shortfall.
        """

        self._validate_confidence(confidence)
        self._validate_horizon(horizon)

        returns = self._calculate_returns(1)

        covariance_matrix = returns.cov()

        aligned_sensitivities = self.sensitivities[
            covariance_matrix.columns
        ]

        portfolio_variance = (
            aligned_sensitivities.T
            @ covariance_matrix
            @ aligned_sensitivities
        )

        portfolio_volatility = np.sqrt(
            portfolio_variance
        )

        horizon_volatility = (
            portfolio_volatility * np.sqrt(horizon)
        )

        z_score = norm.ppf(1 - confidence)

        es = (
            horizon_volatility
            * norm.pdf(z_score)
            / (1 - confidence)
        )

        return es

    # ---------------------------------------------------------
    # Monte Carlo VaR
    # ---------------------------------------------------------

    def monte_carlo_var(
        self,
        confidence=0.95,
        horizon=1,
        simulations=10000,
        seed=42
    ):
        """
        Calculate Monte Carlo VaR.

        Risk-factor returns are simulated from a multivariate
        normal distribution using the historical mean vector and
        covariance matrix.

        Parameters
        ----------
        confidence : float, default=0.95
            VaR confidence level.

        horizon : int, default=1
            Return horizon in periods.

        simulations : int, default=10000
            Number of Monte Carlo simulations.

        seed : int, default=42
            Random-number-generator seed used for reproducibility.

        Returns
        -------
        float
            Monte Carlo VaR.

        Notes
        -----
        For horizons greater than one period, simulated returns
        are scaled using the square-root-of-time assumption.
        """

        self._validate_confidence(confidence)
        self._validate_horizon(horizon)
        self._validate_simulations(simulations)

        returns = self._calculate_returns(1)

        mean_returns = returns.mean()
        covariance_matrix = returns.cov()

        rng = np.random.default_rng(seed)

        simulated_returns = rng.multivariate_normal(
            mean=mean_returns,
            cov=covariance_matrix,
            size=simulations
        )

        if horizon > 1:
            simulated_returns *= np.sqrt(horizon)

        simulated_returns = pd.DataFrame(
            simulated_returns,
            columns=returns.columns
        )

        simulated_pnl = self._calculate_portfolio_pnl(
            simulated_returns
        )

        var_threshold = simulated_pnl.quantile(
            1 - confidence
        )

        var = abs(var_threshold)

        return var

    # ---------------------------------------------------------
    # Monte Carlo Expected Shortfall
    # ---------------------------------------------------------

    def monte_carlo_es(
        self,
        confidence=0.95,
        horizon=1,
        simulations=10000,
        seed=42
    ):
        """
        Calculate Monte Carlo Expected Shortfall.

        Expected Shortfall is calculated as the average loss
        beyond the simulated VaR threshold.

        Parameters
        ----------
        confidence : float, default=0.95
            ES confidence level.

        horizon : int, default=1
            Return horizon in periods.

        simulations : int, default=10000
            Number of Monte Carlo simulations.

        seed : int, default=42
            Random-number-generator seed used for reproducibility.

        Returns
        -------
        float
            Monte Carlo Expected Shortfall.
        """

        self._validate_confidence(confidence)
        self._validate_horizon(horizon)
        self._validate_simulations(simulations)

        returns = self._calculate_returns(1)

        mean_returns = returns.mean()
        covariance_matrix = returns.cov()

        rng = np.random.default_rng(seed)

        simulated_returns = rng.multivariate_normal(
            mean=mean_returns,
            cov=covariance_matrix,
            size=simulations
        )

        if horizon > 1:
            simulated_returns *= np.sqrt(horizon)

        simulated_returns = pd.DataFrame(
            simulated_returns,
            columns=returns.columns
        )

        simulated_pnl = self._calculate_portfolio_pnl(
            simulated_returns
        )

        var_threshold = simulated_pnl.quantile(
            1 - confidence
        )

        tail_losses = simulated_pnl[
            simulated_pnl <= var_threshold
        ]

        if tail_losses.empty:
            raise ValueError(
                "No observations available in the Expected "
                "Shortfall tail."
            )

        es = abs(tail_losses.mean())

        return es

    # ---------------------------------------------------------
    # Basel-Oriented Expected Shortfall
    # ---------------------------------------------------------

    def basel_es(self):
        """
        Calculate Basel-oriented Expected Shortfall.

        This method applies the Basel FRTB Expected Shortfall
        confidence level of 97.5% using a 10-day base horizon.

        This is an educational Basel-oriented implementation.
        It does not implement the full Basel liquidity-horizon
        scaling, stressed ES calibration, reduced risk-factor
        set, or regulatory capital aggregation framework.

        Returns
        -------
        float
            Historical Expected Shortfall at 97.5% confidence
            over a 10-day base horizon.
        """

        confidence = 0.975
        horizon = 10

        es = self.historical_es(
            confidence=confidence,
            horizon=horizon
        )

        return es

    # ---------------------------------------------------------
    # Stress Loss
    # ---------------------------------------------------------

    def stress_loss(self, stress_return):
        """
        Calculate portfolio loss under a common stress shock.

        Parameters
        ----------
        stress_return : float
            Hypothetical return shock applied to every risk factor.

        Returns
        -------
        float
            Absolute portfolio loss under the stress scenario.

        Notes
        -----
        The current implementation applies the same shock to every
        risk factor. Factor-specific stress scenarios can be added
        in a future version.
        """

        if not isinstance(stress_return, (int, float)):
            raise ValueError(
                "stress_return must be numeric."
            )

        if not np.isfinite(stress_return):
            raise ValueError(
                "stress_return must be finite."
            )

        stress_returns = pd.Series(
            stress_return,
            index=self.risk_factor_prices.columns
        )

        pnl = stress_returns @ self.sensitivities

        return abs(pnl)

    # ---------------------------------------------------------
    # Main Calculation Interface
    # ---------------------------------------------------------

    def calculate(
        self,
        confidence=0.95,
        horizon=1,
        methodology="historical",
        simulations=10000,
        seed=42
    ):
        """
        Calculate VaR and Expected Shortfall using the selected method.

        Parameters
        ----------
        confidence : float, default=0.95
            Risk confidence level.

        horizon : int, default=1
            Risk horizon in periods.

        methodology : str, default="historical"
            Risk methodology. Supported values are:
            "historical", "parametric", and "monte_carlo".

        simulations : int, default=10000
            Number of simulations used for Monte Carlo methods.

        seed : int, default=42
            Random-number-generator seed used for Monte Carlo methods.

        Returns
        -------
        dict
            Dictionary containing methodology, confidence, horizon,
            VaR, and Expected Shortfall.

        Examples
        --------
        result = engine.calculate(
            confidence=0.99,
            horizon=10,
            methodology="historical"
        )
        """

        self._validate_confidence(confidence)
        self._validate_horizon(horizon)
        self._validate_simulations(simulations)

        if not isinstance(methodology, str):
            raise ValueError(
                "Methodology must be a string: "
                "historical, parametric, or monte_carlo."
            )

        methodology = methodology.lower()

        if methodology == "historical":

            var = self.historical_var(
                confidence=confidence,
                horizon=horizon
            )

            es = self.historical_es(
                confidence=confidence,
                horizon=horizon
            )

        elif methodology == "parametric":

            var = self.parametric_var(
                confidence=confidence,
                horizon=horizon
            )

            es = self.parametric_es(
                confidence=confidence,
                horizon=horizon
            )

        elif methodology == "monte_carlo":

            var = self.monte_carlo_var(
                confidence=confidence,
                horizon=horizon,
                simulations=simulations,
                seed=seed
            )

            es = self.monte_carlo_es(
                confidence=confidence,
                horizon=horizon,
                simulations=simulations,
                seed=seed
            )

        else:

            raise ValueError(
                "Methodology must be one of: "
                "historical, parametric, monte_carlo."
            )

        return {
            "methodology": methodology,
            "confidence": confidence,
            "horizon": horizon,
            "VaR": round(var, 2),
            "Expected Shortfall": round(es, 2)
        }

In [11]:
engine = RiskEngine(
    sensitivities=sensitivities,
    risk_factor_prices=risk_factor_prices
)

print("Risk engine created successfully.")

Risk engine created successfully.


In [12]:
engine.historical_var(confidence=0.95, horizon=1)

np.float64(4667.019507196981)

In [13]:
engine.historical_var(confidence=0.99, horizon=1)

np.float64(4773.871496419709)

In [15]:
engine.historical_var(confidence=0.99, horizon=10)

ValueError: Insufficient historical observations for a 10-period horizon.

In [16]:
engine.historical_es(
    confidence=0.95,
    horizon=1
)

np.float64(4800.5844937253905)

In [17]:
engine.historical_es(
    confidence=0.99,
    horizon=1
)

np.float64(4800.5844937253905)

In [18]:
engine.parametric_var(
    confidence=0.95,
    horizon=1
)

np.float64(11448.797014315212)

In [19]:
engine.parametric_var(
    confidence=0.99,
    horizon=1
)

np.float64(16192.252099622901)

In [20]:
engine.parametric_es(
    confidence=0.95,
    horizon=1
)

np.float64(14357.25334159323)

In [21]:
engine.parametric_es(
    confidence=0.99,
    horizon=1
)

np.float64(18550.88872859673)

In [22]:
engine.monte_carlo_var(
    confidence=0.95,
    horizon=1
)

np.float64(10024.717281548044)

In [23]:
engine.monte_carlo_var(
    confidence=0.99,
    horizon=1
)

np.float64(14398.98165469149)

In [24]:
engine.monte_carlo_es(
    confidence=0.95,
    horizon=1
)

np.float64(12796.10119517614)

In [25]:
engine.monte_carlo_es(
    confidence=0.99,
    horizon=1
)

np.float64(16786.29092198568)

In [26]:
result = engine.calculate(
    confidence=0.95,
    horizon=1,
    methodology="historical"
)

result

{'methodology': 'historical',
 'confidence': 0.95,
 'horizon': 1,
 'VaR': np.float64(4667.02),
 'Expected Shortfall': np.float64(4800.58)}

In [27]:
engine.calculate(
    confidence=0.95,
    horizon=1,
    methodology="parametric"
)

{'methodology': 'parametric',
 'confidence': 0.95,
 'horizon': 1,
 'VaR': np.float64(11448.8),
 'Expected Shortfall': np.float64(14357.25)}

In [28]:
engine.calculate(
    confidence=0.95,
    horizon=1,
    methodology="monte_carlo"
)

{'methodology': 'monte_carlo',
 'confidence': 0.95,
 'horizon': 1,
 'VaR': np.float64(10024.72),
 'Expected Shortfall': np.float64(12796.1)}

In [29]:
engine.stress_loss(
    stress_return=-0.05
)

np.float64(75000.0)

In [30]:
print(engine.calculate(
    confidence=0.95,
    horizon=1,
    methodology="historical"
))

{'methodology': 'historical', 'confidence': 0.95, 'horizon': 1, 'VaR': np.float64(4667.02), 'Expected Shortfall': np.float64(4800.58)}


In [31]:
print(engine.calculate(
    confidence=0.95,
    horizon=1,
    methodology="parametric"
))

{'methodology': 'parametric', 'confidence': 0.95, 'horizon': 1, 'VaR': np.float64(11448.8), 'Expected Shortfall': np.float64(14357.25)}


In [32]:
print(engine.calculate(
    confidence=0.95,
    horizon=1,
    methodology="monte_carlo"
))

{'methodology': 'monte_carlo', 'confidence': 0.95, 'horizon': 1, 'VaR': np.float64(10024.72), 'Expected Shortfall': np.float64(12796.1)}


In [33]:
engine.calculate(
    confidence=1.5,
    horizon=1,
    methodology="historical"
)

ValueError: Confidence must be between 0 and 1.

In [34]:
engine.calculate(
    confidence=0.95,
    horizon=0,
    methodology="historical"
)

ValueError: Horizon must be a positive integer.

In [35]:
print(
    engine.calculate(
        confidence=0.95,
        horizon=1,
        methodology="historical"
    )
)

print(
    engine.calculate(
        confidence=0.95,
        horizon=1,
        methodology="parametric"
    )
)

print(
    engine.calculate(
        confidence=0.95,
        horizon=1,
        methodology="monte_carlo"
    )
)

{'methodology': 'historical', 'confidence': 0.95, 'horizon': 1, 'VaR': np.float64(4667.02), 'Expected Shortfall': np.float64(4800.58)}
{'methodology': 'parametric', 'confidence': 0.95, 'horizon': 1, 'VaR': np.float64(11448.8), 'Expected Shortfall': np.float64(14357.25)}
{'methodology': 'monte_carlo', 'confidence': 0.95, 'horizon': 1, 'VaR': np.float64(10024.72), 'Expected Shortfall': np.float64(12796.1)}


In [36]:
print("Input validation passed.")

Input validation passed.


In [37]:
bad_sensitivities = pd.Series({
    "EURUSD": 1_000_000,
    "USDJPY": 500_000
})

bad_engine = RiskEngine(
    sensitivities=bad_sensitivities,
    risk_factor_prices=risk_factor_prices
)

ValueError: Risk factors in sensitivities and risk_factor_prices must match.

In [38]:
engine.calculate(
    confidence=0.95,
    horizon=1,
    methodology="monte_carlo",
    simulations=0
)

ValueError: Simulations must be a positive integer.

In [39]:
engine.calculate(
    confidence=0.95,
    horizon=1,
    methodology="monte_carlo",
    simulations=-1000
)

ValueError: Simulations must be a positive integer.

In [40]:
engine.calculate(
    confidence=0.95,
    horizon=1,
    methodology="monte_carlo",
    simulations=5000.5
)

ValueError: Simulations must be a positive integer.

In [41]:
engine.calculate(
    confidence=0.95,
    horizon=1,
    methodology="something_else"
)

ValueError: Methodology must be one of: historical, parametric, monte_carlo.

In [42]:
test_cases = [
    (0.95, 1, "historical"),
    (0.99, 1, "historical"),
    (0.95, 10, "historical"),
    (0.99, 10, "historical"),

    (0.95, 1, "parametric"),
    (0.99, 1, "parametric"),
    (0.95, 10, "parametric"),
    (0.99, 10, "parametric"),

    (0.95, 1, "monte_carlo"),
    (0.99, 1, "monte_carlo"),
    (0.95, 10, "monte_carlo"),
    (0.99, 10, "monte_carlo")
]

for confidence, horizon, methodology in test_cases:

    result = engine.calculate(
        confidence=confidence,
        horizon=horizon,
        methodology=methodology
    )

    print(
        f"{methodology:12} | "
        f"Confidence: {confidence:.0%} | "
        f"Horizon: {horizon} day(s) | "
        f"VaR: {result['VaR']:,.2f} | "
        f"ES: {result['Expected Shortfall']:,.2f}"
    )

historical   | Confidence: 95% | Horizon: 1 day(s) | VaR: 4,667.02 | ES: 4,800.58
historical   | Confidence: 99% | Horizon: 1 day(s) | VaR: 4,773.87 | ES: 4,800.58


ValueError: Insufficient historical observations for a 10-period horizon.

In [43]:
test_cases = [
    ("historical", 0.95, 1),
    ("historical", 0.99, 1),
    ("parametric", 0.95, 1),
    ("parametric", 0.99, 1),
    ("monte_carlo", 0.95, 1),
    ("monte_carlo", 0.99, 1),
]

for methodology, confidence, horizon in test_cases:

    result = engine.calculate(
        confidence=confidence,
        horizon=horizon,
        methodology=methodology
    )

    print(
        f"{methodology:12} | "
        f"Confidence: {confidence:.0%} | "
        f"Horizon: {horizon} day | "
        f"VaR: {result['VaR']:,.2f} | "
        f"ES: {result['Expected Shortfall']:,.2f}"
    )

historical   | Confidence: 95% | Horizon: 1 day | VaR: 4,667.02 | ES: 4,800.58
historical   | Confidence: 99% | Horizon: 1 day | VaR: 4,773.87 | ES: 4,800.58
parametric   | Confidence: 95% | Horizon: 1 day | VaR: 11,448.80 | ES: 14,357.25
parametric   | Confidence: 99% | Horizon: 1 day | VaR: 16,192.25 | ES: 18,550.89
monte_carlo  | Confidence: 95% | Horizon: 1 day | VaR: 10,024.72 | ES: 12,796.10
monte_carlo  | Confidence: 99% | Horizon: 1 day | VaR: 14,398.98 | ES: 16,786.29


In [44]:
engine.calculate(
    confidence=0.95,
    horizon=10,
    methodology="historical"
)

ValueError: Insufficient historical observations for a 10-period horizon.

In [45]:
stress_loss = engine.stress_loss(-0.05)

print(f"Stress Loss: {stress_loss:,.2f}")


Stress Loss: 75,000.00


In [46]:
engine.calculate(
    confidence=0.99,
    horizon=252,
    methodology="historical"
)

ValueError: Insufficient historical observations for a 252-period horizon.

FINAL ENGINE TEST — REAL EUR/USD DATA

In [47]:
eurusd_data = yf.download(
    "EURUSD=X",
    period="5y",
    auto_adjust=False,
    progress=False
)

eurusd_prices = eurusd_data["Close"].dropna()

eurusd_prices.head()

Ticker,EURUSD=X
Date,
2021-11-02,1.159958
2021-11-03,1.157943
2021-11-04,1.161575
2021-11-05,1.155535
2021-11-08,1.156885


In [48]:
eurusd_prices = eurusd_data["Close"].squeeze().dropna()

In [49]:
risk_factor_prices_real = pd.DataFrame({
    "EURUSD": eurusd_prices
})

In [50]:
risk_factor_prices_real.head()

,EURUSD
Date,
2021-11-02,1.159958
2021-11-03,1.157943
2021-11-04,1.161575
2021-11-05,1.155535
2021-11-08,1.156885


In [51]:
sensitivities_real = pd.Series({
    "EURUSD": 10_000_000
})

In [52]:
real_engine = RiskEngine(
    sensitivities=sensitivities_real,
    risk_factor_prices=risk_factor_prices_real
)

In [53]:
result_1d = real_engine.calculate(
    confidence=0.99,
    horizon=1,
    methodology="historical"
)

result_1d

{'methodology': 'historical',
 'confidence': 0.99,
 'horizon': 1,
 'VaR': np.float64(123116.11),
 'Expected Shortfall': np.float64(147247.42)}

In [54]:
test_cases = [
    ("historical", 0.95, 1),
    ("historical", 0.99, 1),
    ("historical", 0.95, 10),
    ("historical", 0.99, 10),
    ("historical", 0.95, 21),
    ("historical", 0.99, 21),
    ("historical", 0.95, 252),
    ("historical", 0.99, 252),

    ("parametric", 0.95, 1),
    ("parametric", 0.99, 1),
    ("parametric", 0.95, 10),
    ("parametric", 0.99, 10),
    ("parametric", 0.95, 21),
    ("parametric", 0.99, 21),
    ("parametric", 0.95, 252),
    ("parametric", 0.99, 252),

    ("monte_carlo", 0.95, 1),
    ("monte_carlo", 0.99, 1),
    ("monte_carlo", 0.95, 10),
    ("monte_carlo", 0.99, 10),
    ("monte_carlo", 0.95, 21),
    ("monte_carlo", 0.99, 21),
    ("monte_carlo", 0.95, 252),
    ("monte_carlo", 0.99, 252),
]


for methodology, confidence, horizon in test_cases:

    result = real_engine.calculate(
        confidence=confidence,
        horizon=horizon,
        methodology=methodology,
        simulations=10000,
        seed=42
    )

    print(
        f"{methodology:12} | "
        f"Confidence: {confidence:.0%} | "
        f"Horizon: {horizon:3} day(s) | "
        f"VaR: €{result['VaR']:,.2f} | "
        f"ES: €{result['Expected Shortfall']:,.2f}"
    )

historical   | Confidence: 95% | Horizon:   1 day(s) | VaR: €74,163.42 | ES: €102,759.99
historical   | Confidence: 99% | Horizon:   1 day(s) | VaR: €123,116.11 | ES: €147,247.42
historical   | Confidence: 95% | Horizon:  10 day(s) | VaR: €218,795.55 | ES: €292,985.19
historical   | Confidence: 99% | Horizon:  10 day(s) | VaR: €333,167.04 | ES: €407,520.56
historical   | Confidence: 95% | Horizon:  21 day(s) | VaR: €306,286.33 | ES: €376,600.07
historical   | Confidence: 99% | Horizon:  21 day(s) | VaR: €424,482.71 | ES: €482,378.29
historical   | Confidence: 95% | Horizon: 252 day(s) | VaR: €614,762.72 | ES: €920,283.81
historical   | Confidence: 99% | Horizon: 252 day(s) | VaR: €1,314,456.55 | ES: €1,427,422.51
parametric   | Confidence: 95% | Horizon:   1 day(s) | VaR: €77,889.18 | ES: €97,676.17
parametric   | Confidence: 99% | Horizon:   1 day(s) | VaR: €110,160.15 | ES: €126,206.57
parametric   | Confidence: 95% | Horizon:  10 day(s) | VaR: €246,307.20 | ES: €308,879.17
parametri

ENGINE VALIDATION TESTS

In [55]:
try:

    real_engine.calculate(
        confidence=1.5,
        horizon=1,
        methodology="historical"
    )

except ValueError as e:

    print("PASS:", e)

PASS: Confidence must be between 0 and 1.


In [56]:
try:

    real_engine.calculate(
        confidence=0.99,
        horizon=0,
        methodology="historical"
    )

except ValueError as e:

    print("PASS:", e)

PASS: Horizon must be a positive integer.


In [57]:
try:

    real_engine.calculate(
        confidence=0.99,
        horizon=1,
        methodology="invalid"
    )

except ValueError as e:

    print("PASS:", e)

PASS: Methodology must be one of: historical, parametric, monte_carlo.


In [58]:
try:

    real_engine.calculate(
        confidence=0.99,
        horizon=1,
        methodology="monte_carlo",
        simulations=0
    )

except ValueError as e:

    print("PASS:", e)

PASS: Simulations must be a positive integer.


In [59]:
bad_sensitivities = pd.Series({
    "GBPUSD": 10_000_000
})

try:

    RiskEngine(
        sensitivities=bad_sensitivities,
        risk_factor_prices=risk_factor_prices_real
    )

except ValueError as e:

    print("PASS:", e)

PASS: Risk factors in sensitivities and risk_factor_prices must match.


In [60]:
try:

    real_engine.calculate(
        confidence=0.99,
        horizon=5000,
        methodology="historical"
    )

except ValueError as e:

    print("PASS:", e)

PASS: Insufficient historical observations for a 5000-period horizon.


In [61]:
mc_test_1 = real_engine.calculate(
    confidence=0.99,
    horizon=10,
    methodology="monte_carlo",
    simulations=10000,
    seed=42
)

mc_test_2 = real_engine.calculate(
    confidence=0.99,
    horizon=10,
    methodology="monte_carlo",
    simulations=10000,
    seed=42
)

print("First result :", mc_test_1)
print("Second result:", mc_test_2)

print(
    "PASS: Reproducible"
    if mc_test_1 == mc_test_2
    else "FAIL: Results differ"
)

First result : {'methodology': 'monte_carlo', 'confidence': 0.99, 'horizon': 10, 'VaR': np.float64(355921.2), 'Expected Shortfall': np.float64(408503.5)}
Second result: {'methodology': 'monte_carlo', 'confidence': 0.99, 'horizon': 10, 'VaR': np.float64(355921.2), 'Expected Shortfall': np.float64(408503.5)}
PASS: Reproducible


In [62]:
for i, code in enumerate(_ih):
    if "class RiskEngine:" in code:
        print("RiskEngine class found in Colab cell:", i)

RiskEngine class found in Colab cell: 10
RiskEngine class found in Colab cell: 62


In [63]:
risk_engine_code = _ih[10]

with open("RiskEngine.py", "w") as f:
    f.write(risk_engine_code)

print("RiskEngine.py created successfully.")

RiskEngine.py created successfully.


In [64]:
import os

print("File exists:", os.path.exists("RiskEngine.py"))
print("File size:", os.path.getsize("RiskEngine.py"), "bytes")

File exists: True
File size: 21281 bytes


In [65]:
for i, code in enumerate(_ih):
    if "class RiskEngine:" in code:
        print(f"Cell {i}: {len(code)} characters")

Cell 10: 21281 characters
Cell 62: 124 characters
Cell 65: 117 characters


In [66]:
risk_engine_code = _ih[10]

with open("RiskEngine.py", "w") as f:
    f.write(risk_engine_code)

print("RiskEngine.py saved.")
print("File size:", os.path.getsize("RiskEngine.py"), "bytes")

RiskEngine.py saved.
File size: 21281 bytes


In [67]:
# from google.colab import files

# files.download("RiskEngine.py")